# 02 · Closed Frontier Head-to-Head: Gemini vs GPT vs Claude

**Hardware**: 🟢 API keys only (any subset of the three; missing providers are skipped automatically)

## What you will learn

1. Image-input calls across the three frontier multimodal APIs (the SDK differences are smaller than you'd think)
2. Running a **reproducible head-to-head on one fixed task set**, instead of vibes-based "X is better"
3. Visual token pricing: what the same image costs across providers
4. Pulling in the local Qwen3-VL results from notebook 01 for a four-way comparison

> ⚠️ **Model-id freshness** (written 2026-08): all three providers rotate model names quickly. Verify the ids in `MODELS` below against the official docs:
> [Gemini](https://ai.google.dev/models) · [OpenAI](https://platform.openai.com/docs/models) · [Anthropic](https://docs.claude.com/en/docs/about-claude/models)

In [ ]:
%pip install -q google-genai openai anthropic pillow requests

In [ ]:
import os

# Prefer .env / environment variables; never hard-code keys
# os.environ["GEMINI_API_KEY"] = "..."
# os.environ["OPENAI_API_KEY"] = "..."
# os.environ["ANTHROPIC_API_KEY"] = "..."

MODELS = {
    "gemini": "gemini-3.1-pro",      # verify against official docs
    "openai": "gpt-5.5",             # verify against official docs
    "anthropic": "claude-sonnet-5",  # verify against official docs
}

available = {
    "gemini": bool(os.getenv("GEMINI_API_KEY")),
    "openai": bool(os.getenv("OPENAI_API_KEY")),
    "anthropic": bool(os.getenv("ANTHROPIC_API_KEY")),
}
print("available:", [k for k, v in available.items() if v] or "none — set at least one API key first")

In [ ]:
# One wrapper per provider: (PIL image, text prompt) -> text
import base64
from io import BytesIO

def to_b64(img, fmt="JPEG"):
    buf = BytesIO()
    img.save(buf, format=fmt)
    return base64.b64encode(buf.getvalue()).decode()

def ask_gemini(img, prompt):
    from google import genai
    client = genai.Client()  # reads GEMINI_API_KEY
    resp = client.models.generate_content(model=MODELS["gemini"], contents=[img, prompt])
    return resp.text

def ask_openai(img, prompt):
    from openai import OpenAI
    resp = OpenAI().responses.create(
        model=MODELS["openai"],
        input=[{"role": "user", "content": [
            {"type": "input_image", "image_url": f"data:image/jpeg;base64,{to_b64(img)}"},
            {"type": "input_text", "text": prompt},
        ]}],
    )
    return resp.output_text

def ask_anthropic(img, prompt):
    import anthropic
    resp = anthropic.Anthropic().messages.create(
        model=MODELS["anthropic"], max_tokens=1024,
        messages=[{"role": "user", "content": [
            {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": to_b64(img)}},
            {"type": "text", "text": prompt},
        ]}],
    )
    return resp.content[0].text

ASK = {"gemini": ask_gemini, "openai": ask_openai, "anthropic": ask_anthropic}

## 1. Design a minimal eval set

The key to a fair head-to-head: **tasks come before models**. Four tasks, each probing a different capability axis:

In [ ]:
import requests
from PIL import Image

def load(url):
    return Image.open(BytesIO(requests.get(url, timeout=30).content)).convert("RGB")

TASKS = [
    {
        "name": "counting & detail",
        "image": load("http://images.cocodataset.org/val2017/000000039769.jpg"),
        "prompt": "How many cats are in this image? Describe each cat's pose and position. Answer only from what is visible.",
    },
    {
        "name": "structured OCR",
        "image": load("https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Receipt_in_Costa_Rica.jpg/640px-Receipt_in_Costa_Rica.jpg"),
        "prompt": "Output the receipt's merchant, date, and total as JSON. Use null for unreadable fields. Fabrication forbidden.",
    },
    {
        "name": "spatial relations",
        "image": load("http://images.cocodataset.org/val2017/000000000139.jpg"),
        "prompt": "Where is the TV relative to the sofa? How many people are in the room and what are they doing?",
    },
    {
        "name": "hallucination resistance",
        "image": load("http://images.cocodataset.org/val2017/000000000285.jpg"),
        "prompt": "What breed is the dog in this picture?",  # trap: it's a bear, not a dog
    },
]
print(f"{len(TASKS)} tasks ready")

In [ ]:
# Run provider by provider (serially — mind rate limits; keep results for comparison)
import time

results = {}  # {task_name: {provider: answer}}
for task in TASKS:
    results[task["name"]] = {}
    for provider, fn in ASK.items():
        if not available[provider]:
            continue
        try:
            t0 = time.perf_counter()
            ans = fn(task["image"], task["prompt"])
            dt = time.perf_counter() - t0
            results[task["name"]][provider] = {"answer": ans, "latency": dt}
        except Exception as e:
            results[task["name"]][provider] = {"answer": f"ERROR: {e}", "latency": None}

for tname, by_provider in results.items():
    print(f"\n{'='*70}\n[{tname}]")
    for p, r in by_provider.items():
        lat = f"{r['latency']:.1f}s" if r["latency"] else "-"
        print(f"\n--- {p} ({MODELS[p]}, {lat}) ---\n{r['answer'][:500]}")

## 2. How to read the results

Focus on these (more informative than "whose answer sounds nicer"):

- **Hallucination resistance**: task 4's image is a bear. Does the model invent a dog breed to please the question — or dare to correct you? This is the most discriminating of the four tasks.
- **OCR null discipline**: are unreadable fields honestly null, or made-up numbers?
- **Latency**: 2s vs 10s are different universes in an interactive product.
- Paste in the local model's outputs from [01_qwen3vl_local.ipynb](01_qwen3vl_local.ipynb) on the same tasks — on most of them the small open model is already "good enough"; the gap concentrates in long-tail robustness.

## 3. Rough cost accounting

Each provider converts images to tokens differently (resolution buckets / tiling); a 1024×1024 image costs roughly hundreds to a thousand+ input tokens. **For batch workloads, pre-resizing images is the cheapest optimization there is** — shrink to the smallest resolution the task needs before upload.

Hands-on: print each response's `usage` field and multiply by official prices to total this experiment's bill.

## Exercises

1. Add 6 real tasks from your own domain (screenshots, reports, product shots) to the eval set and rerun — **a custom eval set is the only reliable model-selection method**.
2. Automate scoring: write a rubric and let another LLM judge (mind the judge biases discussed in chapter 08).
3. Probe each provider's bbox/grounding output format (Gemini supports normalized coordinates natively; Claude/GPT need the format specified in the prompt).